# 4 Model Building 
# 4.2 Supervised Learning 
# Task 2: Retrieval-Augmented Generation (RAG)

# Implimenting Rag system on IMDB movie reviews
Model used : "gemini-1.5-flash"

# Import and Clean the dataset

In [13]:
#pip install nltk

In [1]:
import pandas as pd
import numpy as np

In [5]:
df_data=pd.read_csv('IMDB Dataset.csv')
print(df_data.shape)
print(df_data.head())

(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [7]:
df_review=df_data['review']
df_sent=df_data['sentiment']

In [15]:
def lowercase_text(text):
    # Convert text to lowercase.
    return str(text).lower()
df_review=df_review.apply(lowercase_text)
import re
def replace_urls(text):
    """
    Replace URLs in text with a placeholder.
    In NLP, we might want to remove URLs to:
        Reduce noise - URLs rarely add semantic meaning.
        Improve model performance - Less irrelevant data = better learning.
        Normalize text - Keeps data consistent and clean.
    """
    # Regular expression pattern to match URLs
    # This pattern matches:
    # - http:// or https:// at the beginning
    # - Followed by any combination of:
    #   - Letters (a-z, A-Z)
    #   - Numbers (0-9)
    #   - Special characters like $-_@.&+
    #   - Escaped characters like !*()
    #   - URL-encoded characters (e.g., %20 for space)
    # The + at the end ensures we match one or more of these characters
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'

    # Replace URLs with placeholder
    return re.sub(url_pattern, 'url', str(text))

# Apply URL replacement to all summaries
df_review = df_review.apply(replace_urls)
emoji_pattern = re.compile("["

    # Emoticons (e.g., 😀😁😂🤣😃😄😅😆)
    u"\U0001F600-\U0001F64F"

    # Symbols & pictographs (e.g., 🔥🎉💡📦📱)
    u"\U0001F300-\U0001F5FF"

    # Transport & map symbols (e.g., 🚗✈️🚀🚉)
    u"\U0001F680-\U0001F6FF"

    # Flags (e.g., 🇺🇸🇬🇧🇨🇦 — these are pairs of regional indicators)
    u"\U0001F1E0-\U0001F1FF"

    # Dingbats (e.g., ✂️✈️✉️⚽)
    u"\u2700-\u27BF"

    # Supplemental Symbols & Pictographs (e.g., 🤖🥰🧠🦾)
    u"\U0001F900-\U0001F9FF"

    # Symbols & Pictographs Extended-A (e.g., 🪄🪅🪨)
    u"\U0001FA70-\U0001FAFF"

    # Miscellaneous symbols (e.g., ☀️☁️☂️⚡)
    u"\u2600-\u26FF"

    "]+", flags=re.UNICODE)
emoticon_pattern = re.compile(r'(:\)|:|:D|:P|;|:D|:P|;|:-\)|:-D|:-P|:\'\(|:\||:\*)')
def remove_and_print(text):
    if emoji_pattern.search(text) or emoticon_pattern.search(text):
        #print(text)
        text = emoji_pattern.sub('', text)
        text = emoticon_pattern.sub('', text)
        #print(text)
    return text
df_review = df_review.apply(remove_and_print)
def preprocess_data(text):
    #decontraction
    text = re.sub(r"won\'t", " will not", text)
    text = re.sub(r"won\'t've", " will not have", text)
    text = re.sub(r"can\'t", " can not", text)
    text = re.sub(r"don\'t", " do not", text)    
    text = re.sub(r"can\'t've", " can not have", text)
    text = re.sub(r"ma\'am", " madam", text)
    text = re.sub(r"let\'s", " let us", text)
    text = re.sub(r"ain\'t", " am not", text)
    text = re.sub(r"shan\'t", " shall not", text)
    text = re.sub(r"sha\n't", " shall not", text)
    text = re.sub(r"o\'clock", " of the clock", text)
    text = re.sub(r"y\'all", " you all", text)
    text = re.sub(r"n\'t", " not", text)
    text = re.sub(r"n\'t've", " not have", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"\'d", " would", text)
    text = re.sub(r"\'d've", " would have", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'ll've", " will have", text)
    text = re.sub(r"\'t", " not", text)
    text = re.sub(r"\'ve", " have", text)
    text = re.sub(r"\'m", " am", text)
    text = re.sub(r"\'re", " are", text)
    return text
df_review = df_review.apply(preprocess_data)
print(df_review[0])
print(len(df_review[0]))
def replace_usernames(text):
    """
    Replace usernames (words starting with @) with a generic token.

    Usernames are personally identifiable information and not relevant for sentiment analysis.
    Replacing them with a consistent token helps reduce noise and protect privacy.
    """
    # Replace @username with [USER]
    return re.sub(r'@\w+', '[USER]', str(text))

# Apply username replacement to all review 
df_review = df_review.apply(replace_usernames)
print(df_review[0])
print(len(df_review[0]))

def remove_non_alphabets(text):
    """
    Remove non-alphabetic characters from text.

    Non-alphabetic characters like numbers and special symbols like #, $ etc. often add noise
    to text analysis.
    Removing them helps focus on the actual words and their meanings,
    which is more relevant for sentiment analysis.
    """
    # Keep only alphabetic characters and spaces
    return re.sub(r'[^a-zA-Z\s]', ' ', str(text))

# Apply non-alphabet removal to all summaries
df_review = df_review.apply(remove_non_alphabets)
print(df_review[0])
print(len(df_review[0]))

def remove_consecutive_letters(text):
    """
    Remove consecutive repetitions of letters (more than 3) in text.

    In informal text like reviews, people often express emotions by elongating words
    (e.g., 'sooooo good', 'loooove it'). Normalizing these to a maximum of two consecutive
    occurrences helps standardize the text while preserving the emphasis.
    """
    # In the following pattern, 3 is the minimum number of consecutive occurrences of the same letter
    pattern = r'(\w)\1{3,}'

    # Replace with just two occurrences of the letter
    return re.sub(pattern, r'\1\1', str(text))

# Apply consecutive letter removal to all summaries
df_review = df_review.apply(remove_consecutive_letters)
print(df_review[0])
print(len(df_review[0]))

def remove_short_words(text):
    """
    Remove words that are too short (less than 3 characters) from text.

    Very short words like 'a', 'an', 'to', etc. are often not meaningful for sentiment analysis.
    Removing these helps reduce noise and focus on more significant words that carry sentiment.
    """
    # Split the text into words
    words = str(text).split()

    # Filter out words that are less than 3 characters
    filtered_words = [word for word in words if len(word) >= 3]

    # Join the filtered words back into a string
    return ' '.join(filtered_words)

# Apply short word removal to all summaries
df_review = df_review.apply(remove_short_words)
print(df_review[0])
print(len(df_review[0]))

import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
def remove_stopwords(text):
    """
    Remove common stopwords from text.

    Stopwords are common words like 'the', 'and', 'is', etc. that don't carry much meaning
    for sentiment analysis. Removing them helps focus on the more meaningful content words.
    """
    # Get English stopwords
    stop_words = set(stopwords.words('english'))

    # Split the text into words
    words = str(text).split()

    # Filter out stopwords
    filtered_words = [word for word in words if word.lower() not in stop_words]

    # Join the filtered words back into a string
    return ' '.join(filtered_words)

# Apply stopword removal to all summaries
df_review = df_review.apply(remove_stopwords)
print(df_review[0])
print(len(df_review[0]))
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download WordNet if not already downloaded
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Initialize stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger')

# Function to lemmatize text with POS tagging
def lemmatize_text(text):
    """
    Lemmatize text using WordNet lemmatizer with POS tagging.

    Lemmatization reduces words to their base or dictionary form (lemma) while
    considering the part of speech. This is more accurate than stemming as it
    produces actual words.

    Example:
    - "running" -> "run" (verb form)
    - "better" -> "good" (adjective form)
    """
    # Tokenize the text
    words = str(text).split()

    # Lemmatize each word
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

    # Join the lemmatized words back into a string
    return ' '.join(lemmatized_words)

# Apply lemmatization to all summaries
df_review = df_review.apply(lemmatize_text)
print(df_review[0])
print(len(df_review[0]))

one the other reviewers has mentioned that after watching just episode you will hooked they are right this exactly what happened with the first thing that struck about was its brutality and unflinching scenes violence which set right from the word trust this not show for the faint hearted timid this show pulls punches with regards drugs sex violence its hardcore the classic use the word called that the nickname given the oswald maximum security state penitentary focuses mainly emerald city experimental section the prison where all the cells have glass fronts and face inwards privacy not high the agenda city home many aryans muslims gangstas latinos christians italians irish and more scuffles death stares dodgy dealings and shady agreements are never far away would say the main appeal the show due the fact that goes where other shows would not dare forget pretty pictures painted for mainstream audiences forget charm forget romance does not mess around the first episode ever saw struck n

[nltk_data] Downloading package wordnet to C:\Users\dimit/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


one reviewer mentioned watching episode hooked right exactly happened first thing struck brutality unflinching scene violence set right word trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word called nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda city home many aryan muslim gangsta latino christian italian irish scuffle death stare dodgy dealing shady agreement never far away would say main appeal show due fact go show would dare forget pretty picture painted mainstream audience forget charm forget romance mess around first episode ever saw struck nasty surreal could say ready watched developed taste got accustomed high level graphic violence violence injustice crooked guard sold nickel inmate kill order get away well mannered middle class inmate turned prison bitch due lack street skill prison experience watching may become com

# RAG with gemini API KEY

Once we ask the quetion first it does cosine similarity check on all the reivews and then we use the gemini model to generate responses on most 
similar reviews given our query.

In [21]:
#pip install transformers

In [31]:
#pip install torch accelerate safetensors


In [35]:
#pip install sentence_transformers

In [25]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, pipeline
from transformers import BitsAndBytesConfig
import os
import torch
import time
import re
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

C:\Users\dimit\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

# HF token is read from the HF_TOKEN environment variable (see .env.example)
os.environ.setdefault("HF_TOKEN", "")
os.environ["HF_HOME"] = "./my_huggingface_cache"

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [59]:
#pip install tf-keras

In [61]:
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity

In [63]:
if torch.cuda.device_count()>0:
    my_device = "cuda"
    print(f"You have {torch.cuda.device_count()} GPUs available.")
else:
    my_device = "cpu"
    print("You have no GPUs available. Running on CPU.")

You have no GPUs available. Running on CPU.


In [65]:
embeddings_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', token=os.environ["HF_TOKEN"],
                                       cache_folder=os.environ["HF_HOME"], device=my_device)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [67]:
my_text=[]
for i in df_review:
    my_text.append(i)
#print(my_text)

In [69]:
len(my_text)

50000

encode the reviews into my_embeddings so given our query we get the top 10 most similar revirews to that query.

In [74]:
my_sentences = my_text
my_embeddings = embeddings_model.encode(my_sentences)
print(my_embeddings.shape)

(50000, 384)


# Ask a question here , interact with Rag later

In [76]:
# replace the question that you want and it will be based on this quetion the rag system
my_question = "what are the best action movies in your opinion and does the review matter?" 
my_question_embedding = embeddings_model.encode([my_question]) #encode the quetion to compare it to the reviews
similarity_to_question = cosine_similarity(my_question_embedding, my_embeddings).flatten() 
sorted_indices = similarity_to_question.argsort()[::-1]  # Sort in descending order
print(sorted_indices)

[ 6211 25986 20033 ... 47166  9917  3559]


In [78]:
sorted_sentences = [my_sentences[i] for i in sorted_indices]
print("Sorted sentences based on cosine similarity to the question:",my_question)
for i, sentence in enumerate(sorted_sentences[:10]):
    print("-"*100)
    print(f"Sentence {i+1}, similarity: {similarity_to_question[sorted_indices[i]]:.2f}")
    print(sentence)

Sorted sentences based on cosine similarity to the question: what are the best action movies in your opinion and does the review matter?
----------------------------------------------------------------------------------------------------
Sentence 1, similarity: 0.68
first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation
----------------------------------------------------------------------------------------------------
Sentence 2, similarity: 0.67
good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think
------------------

In [80]:
number_of_reviews_tokeep= 3 # how many of reviews we want to keep from the consine similarity 
context = []  # the context we are going to give to gemini RAG model to have interactive discusions.
for review in sorted_sentences[:number_of_reviews_tokeep]:
    context.append(review)

In [90]:
context

['first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation',
 'good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think',
 'one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy film watch film disappointed']

In [106]:
#!{sys.executable} -m pip install google-generativeai


  Obtaining dependency information for google-generativeai from https://files.pythonhosted.org/packages/6e/40/c42ff9ded9f09ec9392879a8e6538a00b2dc185e834a3392917626255419/google_generativeai-0.8.5-py3-none-any.whl.metadata
  Obtaining dependency information for google-ai-generativelanguage==0.6.15 from https://files.pythonhosted.org/packages/7c/a3/67b8a6ff5001a1d8864922f2d6488dc2a14367ceb651bc3f09a947f2f306/google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata
  Obtaining dependency information for google-api-core from https://files.pythonhosted.org/packages/14/4b/ead00905132820b623732b175d66354e9d3e69fcf2a5dcdab780664e7896/google_api_core-2.25.1-py3-none-any.whl.metadata
  Obtaining dependency information for google-api-python-client from https://files.pythonhosted.org/packages/b1/2c/758f415a19a12c3c6d06902794b0dd4c521d912a59b98ab752bba48812df/google_api_python_client-2.176.0-py3-none-any.whl.metadata
  Obtaining dependency information for google-auth>=2.15.0 from https://file

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: C:\Users\dimit\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [108]:
# RAG System with Google Gemini
import google.generativeai as genai
import os

In [ ]:
API_KEY = os.getenv("GOOGLE_API_KEY", "")  # set in your environment (see .env.example)

# Configure Gemini
genai.configure(api_key=API_KEY)

# Initialize the model
model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini RAG System Ready!")

In [112]:
def simple_rag_query(context, question):
    """Simple RAG query using Gemini"""
    
    prompt = f"""Based on the following context, please answer the question concisely.

Context: {context}

Question: {question}

Answer:"""
    
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Error: {e}"

In [114]:
context

['first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation',
 'good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think',
 'one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy film watch film disappointed']

In [116]:
# Takes some time to respond

In [118]:
# If we ask different quetion at this stage we will not get any results since our curent context is based on our first quetion 
# So we should move on with the interactive rag functionality with angain based on our first quetion that we had. 

In [120]:
# Test 1: Basic fact
print("\n--- Test 1: Basic questions on the top similar review ---")

question = "that comedy movies i should watch?"
    
answer = simple_rag_query(context, question)
print(f"Context: {context}")
print(f"Question: {question}")
print(f"Answer: {answer}")


--- Test 1: Basic questions on the top similar review ---
Context: ['first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation', 'good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think', 'one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy film watch film disa

In [122]:
# Test 2: same quetions as previous to find the similar reviews:
print(my_question,"\n")
answer = simple_rag_query(context, my_question)
print(f"Context: {context}")
print(f"Question: {question}")
print(f"Answer: {answer}")

what are the best action movies in your opinion and does the review matter? 

Context: ['first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation', 'good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think', 'one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy f

# Interactive Rag on most similar reviews again 

In [124]:
def interactive_rag(context):
    """Interactive RAG system"""
    
    print("\n Interactive RAG System")
    print("Enter 'quit' to exit")

    print("based on this context:",context)
    # Get context first
    #print("\nFirst, provide the context:")
    #context =  input("Context: ")
    
    if context.lower() == 'quit':
        return
    
    print(f"\nContext set! Now ask questions about: {context[:100]}...")
    
    while True:
        question = input("\nQuestion: ")
        
        if question.lower() == 'quit':
            break
        
        print(" Thinking...")
        answer = simple_rag_query(context, question)
        print(f"Answer: {answer}")

In [126]:
context

['first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation',
 'good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think',
 'one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy film watch film disappointed']

In [128]:
context1=" ".join(context)

In [130]:
context1

'first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy film watch film disappointed'

In [132]:
# Run the function and interact with the model: 
interactive_rag(context1)


 Interactive RAG System
Enter 'quit' to exit
based on this context: first thing noticed movie well everything set quality movie round suppose could love film action liked pure thriller horror movie offer fully fleshed script horror film thought least version ended brilliantly great throughout story felt honest brutal film excellent tight script keep action moving believable character largely believable situation good action interesting plot seen original must say interesting improvement showed movie good cast meaningful acting although would nice see expansion action scene rather background story fun movie watch keep alert time right vote think one best best film seen long time enough action satisfy fan yet plot good really enjoyed film hooked start finish added blood gore brought realistic nature happens front film even tear jerker ending many people think must watch anyone seen many review slating film fair film get bad review turn best prof rent film buy film watch film disappointe


Question:  what defines a good movie


 Thinking...
Answer: Excellent script, believable characters and situations, good action, interesting plot, and meaningful acting.



Question:  q


 Thinking...
Answer: A great, brutal, and thrilling horror film with a tight script, believable characters, and good action.  Highly recommended.



Question:  what movies do you like 


 Thinking...
Answer: Action thrillers, horror movies with fleshed-out scripts, and this particular film (highly praised).



Question:  how is a review constructive


 Thinking...
Answer: Offers specific praise (e.g., tight script, believable characters, good action, interesting plot, excellent ending), balanced with constructive criticism (e.g., expand action scenes).



Question:  quit


In [94]:
len(df_review[0])

1058

In [134]:
#save the df_review dataset into csv file for user interface data 
df_review[:5000].to_csv('reviews2.csv')

In [136]:
text1=pd.read_csv('reviews2.csv')
# Store reviews as a list instead of joining them
reviews_list = text1.review.tolist()
# For initial RAG text, still join first 10 for compatibility
my_initial_rag_text=". ".join(reviews_list[:10])

In [103]:
#my_initial_rag_text